# No.8 CT フィルタード逆投影

X線CT画像の再構成アルゴリズム「フィルタード逆投影（FBP）」をPythonで実装します。

**処理ステップ:**
1. 順方向投影（Radon変換）→ サイノグラム生成
2. ランプフィルタで周波数フィルタリング
3. 逆投影 → CT画像再構成

**C言語版との対応:**
- `projection3.c` → `skimage.transform.radon()`
- `filter3.c` + `backprojection3.c` → `skimage.transform.iradon(filter_name='shepp-logan')`

In [ ]:
import numpy as np
import skimage.io
import skimage.transform
import skimage.data
import matplotlib.pyplot as plt
import japanize_matplotlib


## Part 1: ファントムで原理確認

まずSheppとLoganが設計したMRIテスト用ファントムで、投影→再構成の往復を確認します。

In [ ]:
# Shepp-Logan ファントム（skimage内蔵）
phantom = skimage.data.shepp_logan_phantom()
phantom = skimage.transform.resize(phantom, (256, 256))

angles = np.linspace(0, 180, 256, endpoint=False)

# 順方向投影（Radon変換）
sinogram = skimage.transform.radon(phantom, theta=angles)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Original Phantom')
axes[0].axis('off')
axes[1].imshow(sinogram, cmap='gray', aspect='auto',
               extent=[0, 180, sinogram.shape[0], 0])
axes[1].set_title('Sinogram (Radon transform)')
axes[1].set_xlabel('Projection angle [deg]')
plt.tight_layout()
plt.show()

In [ ]:
# フィルタード逆投影（FBP）
# filter_name='shepp-logan': C版の sin(pi*v)/pi フィルタと等価
reconstruction = skimage.transform.iradon(sinogram, theta=angles,
                                           filter_name='shepp-logan')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(reconstruction, cmap='gray')
axes[1].set_title('FBP Reconstruction')
axes[1].axis('off')
plt.tight_layout()
plt.show()

error = np.abs(phantom - reconstruction / reconstruction.max())
print(f'最大誤差: {error.max():.4f}')

## Part 2: 実データ（Chest256.dat）

C言語版プログラムが使っていた実際のCTデータを読み込んで再構成します。

In [ ]:
def load_dat(filepath, n_angles=256, n_samples=256):
    sinogram = np.zeros((n_samples, n_angles))
    with open(filepath) as f:
        lines = [l.strip() for l in f if l.strip()]
    idx = 0
    for angle_idx in range(n_angles):
        for sample_idx in range(n_samples):
            if idx < len(lines):
                parts = lines[idx].split()
                if len(parts) >= 2:
                    sinogram[sample_idx, angle_idx] = float(parts[1])
                idx += 1
    return sinogram

sinogram_real = load_dat('data/Chest256.dat')
print(f'サイノグラム shape: {sinogram_real.shape}')

plt.figure(figsize=(8, 4))
plt.imshow(sinogram_real, cmap='gray', aspect='auto')
plt.title('Chest256 Sinogram')
plt.colorbar()
plt.show()

In [ ]:
recon_real = skimage.transform.iradon(sinogram_real, theta=angles,
                                       filter_name='shepp-logan')

plt.figure(figsize=(6, 6))
plt.imshow(recon_real, cmap='gray')
plt.title('Chest CT Reconstruction (FBP)')
plt.axis('off')
plt.show()

# 保存
normalized = recon_real - recon_real.min()
normalized = (normalized / normalized.max() * 255).astype(np.uint8)
skimage.io.imsave('ct_reconstructed.pgm', normalized)
print('ct_reconstructed.pgm を保存しました')